In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark=SparkSession.builder.appName("SmartHomeEnergyTracker").getOrCreate()

In [2]:
df=spark.read.csv("cleaned_energy_usage.csv",header=True,inferSchema=True)
df=df.withColumn("timestamp",to_timestamp(col("timestamp")))
df=df.withColumn("hour",hour(col("timestamp")))
df=df.withColumn("usage_period",when(col("hour")>17,"Peak").otherwise("OffPeak"))
df.show()

+------+---------+----------------+------------+----------+----------------------+-------------------+----+------------+
|log_id|device_id|     device_name|   room_name|energy_kwh|usage_duration_minutes|          timestamp|hour|usage_period|
+------+---------+----------------+------------+----------+----------------------+-------------------+----+------------+
|     1|        1| Air Conditioner| Living Room|       5.6|                   240|2026-05-01 08:00:00|   8|     OffPeak|
|     2|        2|     Ceiling Fan|     Bedroom|      0.85|                   360|2026-05-03 09:00:00|   9|     OffPeak|
|     3|        3|    Refrigerator|     Kitchen|       2.3|                  1440|2026-05-08 10:00:00|  10|     OffPeak|
|     4|        4|        Smart TV| Living Room|       1.2|                   180|2026-05-12 19:00:00|  19|        Peak|
|     5|        5|  Speaker System|Home Theater|      1.95|                   210|2026-05-12 20:00:00|  20|        Peak|
|     6|        6|    Water Heat

In [3]:
peak_offpeak_summary=df.groupBy("device_name","usage_period").agg(round(sum("energy_kwh"),2).alias("total_energy_usage"))
print("Peak vs OffPeak Energy Usage")
peak_offpeak_summary.show()

Peak vs OffPeak Energy Usage
+----------------+------------+------------------+
|     device_name|usage_period|total_energy_usage|
+----------------+------------+------------------+
|        Smart TV|        Peak|               1.2|
|Desktop Computer|        Peak|               2.1|
|  Speaker System|        Peak|              1.95|
| Washing Machine|     OffPeak|               4.8|
|    Dining Light|        Peak|              0.35|
|    Refrigerator|     OffPeak|               2.3|
| Air Conditioner|     OffPeak|               5.6|
|     Ceiling Fan|     OffPeak|              0.85|
|    Water Heater|     OffPeak|               7.5|
+----------------+------------+------------------+



In [4]:
top_devices=df.groupBy("device_name").agg(round(sum("energy_kwh"),2).alias("total_energy_usage")).orderBy(col("total_energy_usage").desc())
print("Top Energy Consuming Devices")
top_devices.show()

Top Energy Consuming Devices
+----------------+------------------+
|     device_name|total_energy_usage|
+----------------+------------------+
|    Water Heater|               7.5|
| Air Conditioner|               5.6|
| Washing Machine|               4.8|
|    Refrigerator|               2.3|
|Desktop Computer|               2.1|
|  Speaker System|              1.95|
|        Smart TV|               1.2|
|     Ceiling Fan|              0.85|
|    Dining Light|              0.35|
+----------------+------------------+



In [5]:
high_usage_devices=df.filter(col("energy_kwh")>4)
print("Devices Consuming More Than 4 kWh")
high_usage_devices.select("device_name","energy_kwh").show()

Devices Consuming More Than 4 kWh
+---------------+----------+
|    device_name|energy_kwh|
+---------------+----------+
|Air Conditioner|       5.6|
|   Water Heater|       7.5|
|Washing Machine|       4.8|
+---------------+----------+



In [6]:
# Save the result
top_devices.toPandas().to_csv("top_devices_output.csv")
peak_offpeak_summary.toPandas().to_csv("peak_offpeak_output.csv")
print("Output Files Generated Successfully")

Output Files Generated Successfully
